### Imports:

In [12]:
import pandas as pd
import math
import numpy as np
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

RANDOMSTATE = 0

In [13]:
import subprocess
subprocess.run(["pip", "list"], capture_output=True, text=True).stdout

'Package                Version\n---------------------- ------------\nantlr4-python3-runtime 4.9.3\nanyio                  4.13.0\nappnope                0.1.4\nasttokens              3.0.1\nbackoff                2.2.1\ncertifi                2026.4.22\ncomm                   0.2.3\ncontourpy              1.3.2\ncycler                 0.12.1\ndebugpy                1.8.20\ndecorator              5.2.1\ndotenv                 0.9.9\nexceptiongroup         1.3.1\nexecuting              2.2.1\nfonttools              4.62.1\nh11                    0.16.0\nh2                     4.3.0\nhpack                  4.1.0\nhttpcore               1.0.9\nhttpx                  0.28.1\nhyperframe             6.1.0\nidna                   3.13\nipykernel              7.2.0\nipython                8.37.0\njedi                   0.19.2\njoblib                 1.5.3\njupyter_client         8.8.0\njupyter_core           5.9.1\nkiwisolver             1.5.0\nmarkdown-it-py         4.0.0\nmatplotlib         

## Algorithms:


In [14]:
from collections import defaultdict


def forest_diffusion_repaint_imputation(dataset, trained_models, n_noise=100, noise_levels=50, r=10, j=5):
    M = dataset.notna().astype(int)
    
    X = pd.DataFrame(
        np.random.normal(0, 1, dataset.shape), 
        columns=dataset.columns, 
        index=dataset.index)
    
    t = 1.0
    h = 1.0 / noise_levels
    repaints = defaultdict(lambda: 0)

    while t > 0:
        t = round(t, 2)
        
        X_next = pd.DataFrame(np.zeros(X.shape), columns=X.columns, index=X.index)
        
        X_rev = reverse(X, t, trained_models[t], noise_levels, flow=False)
        
        Z_known = np.random.normal(0, 1, dataset.shape)
        X_known_noisy, _ = forward(Z_known, dataset, t, flow=False)
        X_next = X_known_noisy.where(M == 1, X_rev)
        
        X = X_next

        if repaints[t] < r and (round(noise_levels * t) % j == 0) and t < 1.0:
            repaints[t] += 1
            Z_jump = np.random.normal(0, 1, dataset.shape)
            t_next = min(round(t + j * h, 2), 1.0)
            
            X_jump = forward_step(Z_jump, X, t, t_next, flow=False)
            X[M == 1] = X_jump[M == 1]
            t = t_next
            continue  
        
        t -= h
        
    return X

def forward(Z, X, t, flow=False, beta=[0.1,20]):
    if flow:
        X = ((1-t) * Z) + (t * X)
        Y = X - Z
    else: 
        C = -1/4*(t**2) * (beta[1]-beta[0]) - (0.5 * t * beta[0])
        X = (np.exp(C) * X) + (np.sqrt(1-np.exp(2*C))*Z)
        Y = Z
    return X, Y

def forward_step(Z, X, t, t_next, flow=False, beta=[0.1,20]):
    h = t_next - t
    if flow:
        X = Z + (h*(X - Z))
    else:
        beta = beta[0] + (t*(beta[1]-beta[0]))
        X = X - h * (0.5 * beta * X + math.sqrt(beta) * Z)
    return X

def reverse(X, t, model, noise_levels, flow=False, beta=[0.1, 20]):
    Z = pd.DataFrame(
        np.random.normal(loc=0.0, scale=1.0, size=X.shape),
        columns=X.columns,
        index=X.index)
    h = 1.0 / noise_levels
    prediction = model.predict(X)

    if flow:
        return X + (h * prediction)
    else:
        beta_t = beta[0] + (t * (beta[1] - beta[0]))
        C = -0.25 * (t**2) * (beta[1] - beta[0]) - (0.5 * t * beta[0])
        score = -prediction / np.sqrt(1 - np.exp(2 * C))
        mu = (-0.5 * beta_t * X) - (beta_t * score)
        
        return X - (h * mu) + (np.sqrt(beta_t * h) * Z)

In [15]:
import os
import joblib

def save_models(trained_models, path="forest_diffusion_models"):
    try:
        os.mkdir(path)
    except FileExistsError:
        pass
    for t, model in trained_models.items():
        filename = os.path.join(path, f'model_t{t:.2f}.joblib')
        joblib.dump(model, filename)
    
def load_models(path="forest_diffusion_models"):
    if not os.path.isdir(path):
        raise FileNotFoundError(f'Model directory {path} not found')
    
    trained_models = {}
    files = sorted(f for f in os.listdir(path))

    if not files:
        raise FileNotFoundError(f'No saved models in dir {path}')
    
    for filename in files:
        t = float(filename.replace("model_t", "").replace(".joblib", ""))
        t = round(t, 2)
        trained_models[t] = joblib.load(os.path.join(path, filename))
    return trained_models


## Forest-diffusion training:

In [16]:
def forest_diffusion_training(dataset, n_noise=100, noise_levels=50, flow=False):
    X_prime = pd.concat([dataset] * n_noise, ignore_index=True)
    Z = np.random.normal(loc=0.0, scale=1.0, size=X_prime.shape)  # Dataframe of gaussian noise, (observations*n_noise) * n_features
    
    noise_schedule = np.linspace(1.0/noise_levels, 1.0, noise_levels) 
    
    trained_models = {}

    for t in noise_schedule:
        t = round(t, 2)
        print(f'Training model for noise level {t}')
        X_t, Y_t = forward(Z, X_prime, t, flow=flow)
        base_model = XGBRegressor(
            n_estimators=100,
            reg_lambda=0, # L2 regularization should be 0 (found in paper)
            reg_alpha=0,  # same for L1
            n_jobs=-1)
        
        model = MultiOutputRegressor(base_model) # to predict all features at once, XGB is single-output
        model.fit(X_t, Y_t)
        trained_models[t] = model
        
    return trained_models
    

## Loading and encoding:

In [17]:
df = pd.read_csv('datasets/train.csv')
def preprocessing(df):
        df[["Deck", "Cabin_num", "Side"]] = df["Cabin"].str.split("/", expand=True)
        df['Cabin_num'] = df['Cabin_num'].astype('Int64')
        df = df.drop(['Cabin', 'Name'], axis=1)

        categorical_features = ["HomePlanet", "Deck", "Side", "Destination"]

        for col in categorical_features:
                dummies = pd.get_dummies(df[col], prefix=col)
                dummies = dummies.where(~df[col].isna(), np.nan)
                df = df.drop(columns=[col]).join(dummies)
        return df

df = pd.read_csv('datasets/train.csv')
kaggle_test = pd.read_csv("datasets/test.csv")
df = preprocessing(df)
kaggle_test = preprocessing(kaggle_test)

df = df.drop(columns=["PassengerId"])
kaggle_test = kaggle_test.drop(columns=["PassengerId"])
df, df_test = train_test_split(df, test_size=0.2, random_state=RANDOMSTATE, 
                                     stratify=df["Transported"])

X_original = df.copy()
df_test_original = df_test.copy()

maxes = df.max().astype('Int64')
mins  = df.min().astype('Int64')

print(len(kaggle_test))

4277


### Min-max normalization [-1, 1]:

In [18]:
kaggle_test["Transported"] = 0
kaggle_test = kaggle_test[df.columns]
kaggle_test_original = kaggle_test.copy()

minmax_scaler = MinMaxScaler(feature_range=(-1, 1))

df = pd.DataFrame(
    minmax_scaler.fit_transform(df),
    columns=df.columns, index=df.index)

df_test = pd.DataFrame(
    minmax_scaler.transform(df_test),
    columns=df_test.columns, index=df_test.index)

kaggle_test = pd.DataFrame(
    minmax_scaler.transform(kaggle_test),
    columns = kaggle_test.columns, index = kaggle_test.index)

original_scaler = minmax_scaler

print(len(kaggle_test))

4277


## Applying repaint to dataset:

In [19]:
#models = load_models() 
models = forest_diffusion_training(df)

Training model for noise level 0.02
Training model for noise level 0.04
Training model for noise level 0.06
Training model for noise level 0.08
Training model for noise level 0.1
Training model for noise level 0.12
Training model for noise level 0.14
Training model for noise level 0.16
Training model for noise level 0.18
Training model for noise level 0.2
Training model for noise level 0.22
Training model for noise level 0.24
Training model for noise level 0.26
Training model for noise level 0.28
Training model for noise level 0.3
Training model for noise level 0.32
Training model for noise level 0.34
Training model for noise level 0.36
Training model for noise level 0.38
Training model for noise level 0.4
Training model for noise level 0.42
Training model for noise level 0.44
Training model for noise level 0.46
Training model for noise level 0.48
Training model for noise level 0.5
Training model for noise level 0.52
Training model for noise level 0.54
Training model for noise level 0.

In [20]:
save_models(models)

In [21]:
X_imputed = forest_diffusion_repaint_imputation(
                    dataset=df, 
                    trained_models=models,
                    n_noise=100,
                    noise_levels=50, 
                    r=10, 
                    j=5)

X_imputed.to_csv("datasets/imputed.csv")

In [22]:
train_column_order = df.columns.tolist()  
kaggle_test = kaggle_test[train_column_order]

kaggle_test_imputed = forest_diffusion_repaint_imputation(
    dataset=kaggle_test,
    trained_models=models,
    n_noise=100,
    noise_levels=50,
    r=10,
    j=5)

print(len(kaggle_test_imputed))

4277


In [23]:
print(len(kaggle_test_imputed))

4277


## Post-processing:

In [25]:

def finalize_imputation(X_imputed, X_original):
    X_clipped = X_imputed.clip(-1, 1)
    X_rescaled = pd.DataFrame(
        original_scaler.inverse_transform(X_clipped),
        columns=X_imputed.columns, index=X_original.index).round()
    
    return X_original.fillna(X_rescaled)

X_final = finalize_imputation(X_imputed, X_original)
kaggle_test = finalize_imputation(kaggle_test_imputed, kaggle_test_original)

M = X_original.isna()
rows, cols = np.where(M)
imputed_df = pd.DataFrame({
    'row': M.index[rows]+2,
    'column': M.columns[cols],
    'imputed_value': X_final.values[rows, cols]
})

imputed_df.to_csv('datasets/Imputed_values.csv')

In [26]:
print(len(kaggle_test))

4277


## Augment dataset with generated data:

In [27]:
def generate_data(models, n_samples, columns, noise_levels=50):
    noise_schedule = [round((n / noise_levels), 2) for n in range(noise_levels, 0, -1)]
    X = pd.DataFrame(np.random.normal(0, 1, (n_samples, len(columns))), columns = columns)

    for t in noise_schedule:
        X = reverse(X, t, models[t], noise_levels)

    return X

def postprocess(X_generated, X_original, mins, maxes):
    X_clipped = X_generated.clip(-1, 1)
    X_rescaled = (((X_clipped + 1) / 2) * (maxes - mins) + mins).round()

    dummy_cols = []
    for c in X_original.columns:
        if any(c.startswith(p) for p in ['HomePlanet_', 'Deck_', 'Side_', 'Destination_']):
            dummy_cols.append(c)
    for group_prefix in ['Deck_', 'HomePlanet_', 'Side_', 'Destination_']:
        cols = [c for c in X_rescaled.columns if c.startswith(group_prefix)]
        X_rescaled[cols] = pd.get_dummies(X_rescaled[cols].idxmax(axis=1)).reindex(columns=cols, fill_value=0)

    for col in ['CryoSleep', 'VIP']:
        if col in X_rescaled.columns:
            X_rescaled[col] = X_rescaled[col].clip(0, 1).round()

    return X_rescaled

In [28]:
X_gen_norm = generate_data(models, len(X_original), columns=df.columns)
print(len(X_gen_norm))

6954


In [29]:
from scipy.stats import rankdata

X_gen = postprocess(X_gen_norm, X_original, mins, maxes) 
print(len(X_gen))

label_probs = X_final['Transported'].value_counts(normalize=True)
label_clf = RandomForestClassifier(n_estimators=100, random_state=42)
feats = [c for c in X_final.columns if c != 'Transported']
label_clf.fit(X_final[feats], X_final['Transported'].astype(float))

probs = label_clf.predict_proba(X_gen[feats])[:, 1]
X_gen['Transported'] = (np.random.rand(len(X_gen)) < probs).astype(float)

X_gen['Transported'] = X_gen['Transported'].astype(bool)
X_final['Transported'] = X_final['Transported'].astype(bool)

spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

cryo_mask = X_gen["CryoSleep"] == 1
X_gen.loc[cryo_mask, spend_cols] = 0.0

def quantile_normalize(input, reference):
    ref_sorted = np.sort(reference)

    ranks = rankdata(input, method="average") - 1
    quantiles = ranks / len(input)

    reference_q_pos = np.linspace(0, 1, len(ref_sorted), endpoint=False)
    remapped = np.interp(quantiles, reference_q_pos, ref_sorted)

    return remapped

awake_generated_mask = X_gen["CryoSleep"] == 0
awake_train_mask = X_final["CryoSleep"] == 0

for col in spend_cols:
    ref = X_final.loc[awake_train_mask, col].dropna().values
    gen_index = X_gen.index[awake_generated_mask]
    X_gen.loc[gen_index, col] = quantile_normalize(X_gen.loc[gen_index, col].values, ref)

print(X_gen.isna().any(axis=None))
X_gen.to_csv("datasets/X_gen.csv")


6954
False


## Make TotalSpending feature

In [31]:
X_final_totalspending = X_final[spend_cols].sum(axis=1) 
X_gen_totalspending = X_gen[spend_cols].sum(axis=1)
kaggle_totalspending = kaggle_test[spend_cols].sum(axis=1)

X_final["TotalSpending"] = X_final_totalspending
X_gen["TotalSpending"] = X_gen_totalspending
kaggle_test["TotalSpending"] = kaggle_totalspending


### Filling test set NA's

In [32]:

X_test_imputed = forest_diffusion_repaint_imputation(
    dataset=df_test,
    trained_models=models,
    noise_levels=50, r=10, j=5)

X_test_final = finalize_imputation(X_test_imputed, df_test_original)
X_test_final['TotalSpending'] = X_test_final[spend_cols].sum(axis=1)

## Prepare for prediction

Reorganizing data to match manually imputed dataset

In [33]:
numerical_features = ["Age","RoomService","FoodCourt","ShoppingMall","Spa","VRDeck","TotalSpending","Cabin_num"]
categorical_features = ["CryoSleep", "VIP", "HomePlanet_Earth","HomePlanet_Europa","HomePlanet_Mars","Deck_A","Deck_B","Deck_C","Deck_D","Deck_E","Deck_F","Deck_G","Deck_T","Side_P","Side_S","Destination_55 Cancri e","Destination_PSO J318.5-22","Destination_TRAPPIST-1e"]

concatenated_dataset = pd.concat([X_gen, X_final], ignore_index=True)
scaler = StandardScaler()
encoder = OneHotEncoder(handle_unknown="ignore")

numerical_data = pd.DataFrame(
    scaler.fit_transform(concatenated_dataset[numerical_features]),
    columns=numerical_features
).reset_index(drop=True)

categorical_data = concatenated_dataset[categorical_features].astype("float")

labels = pd.DataFrame(
    concatenated_dataset["Transported"]
).reset_index(drop=True)

final_dataset = pd.concat([numerical_data, categorical_data, labels], axis=1)

bool_cols = final_dataset.select_dtypes(include='bool').columns
final_dataset[bool_cols] = final_dataset[bool_cols].astype(int)
final_dataset.to_csv("datasets/train_augmented.csv", index=False)

In [34]:
X_test_numerical = pd.DataFrame(
    scaler.transform(X_test_final[numerical_features]),  # transform only, not fit!
    columns=numerical_features,
    index=X_test_final.index
)

X_test_categorical = X_test_final[categorical_features].astype("float")
X_test_labels = X_test_final[["Transported"]]

X_test_augmented = pd.concat([X_test_numerical, X_test_categorical, X_test_labels], axis=1)
bool_cols = X_test_augmented.select_dtypes(include='bool').columns
X_test_augmented[bool_cols] = X_test_augmented[bool_cols].astype(int)

X_test_augmented.to_csv("datasets/X_val_augmented.csv", index=False)

In [35]:
kaggle_numerical = pd.DataFrame(
    scaler.transform(kaggle_test[numerical_features]),  
    columns=numerical_features,index=kaggle_test.index)

kaggle_categorical = kaggle_test[categorical_features].astype("float")
kaggle_labels = kaggle_test[["Transported"]]

kaggle_test = pd.concat([kaggle_numerical, kaggle_categorical, kaggle_labels], axis=1)
bool_cols = kaggle_test.select_dtypes(include='bool').columns
kaggle_test[bool_cols] = kaggle_test[bool_cols].astype(int)

kaggle_test.to_csv("datasets/kaggle_test_processed.csv", index=False)
print(len(kaggle_test))

4277
